# Do-No-Harm Latent Steering: Colab runner (A100)

Push-T + frozen LeRobot Diffusion Policy + stand-in probabilistic WM + Learn-then-Test calibration.

**Code comes from GitHub ([carlo-scr/dnh-latent-steering](https://github.com/carlo-scr/dnh-latent-steering)), outputs go to Drive.**
Edit and `git push` on your laptop, then re-run the setup cell here (it pulls). Data, WMs and results live in
`MyDrive/dnh_runs`, so they survive disconnects.

Select *Runtime → Change runtime type → A100 GPU*, then run top to bottom.

In [42]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing; print("CPUs:", multiprocessing.cpu_count())

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
CPUs: 12


In [43]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/dnh_runs'   # all outputs (data, WMs, results)
os.makedirs(ROOT, exist_ok=True)
os.environ['ROOT'] = ROOT
os.environ['SDL_VIDEODRIVER'] = 'dummy'   # headless pygame
os.environ['MUJOCO_GL'] = 'egl'           # for the later robosuite/robomimic tasks

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Clone (first run) or pull (later runs) the code, then install. Takes ~2-3 min the first time. If pip asks to restart the runtime (torch version change), restart and re-run from the top.

In [44]:
REPO = 'https://github.com/carlo-scr/dnh-latent-steering.git'
CODE = '/content/code'
if os.path.exists(f'{CODE}/.git'):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
%cd {CODE}
!git log --oneline -1
!pip install -q -e .
!python -m pytest -q tests

/content/code
2ca2e5a (HEAD -> main, origin/main, origin/HEAD) Colab runner: clone public repo without a token
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 9.2 MB

## 0. Smoke test
Migrates the hub checkpoint into `ROOT/dp_pusht_migrated` once, checks snapshot/restore, perturbations, K-batched sampling, and paired re-runs.

In [45]:
!python scripts/00_smoke_test.py --ckpt $ROOT/dp_pusht_migrated --n 16 --K 16

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
[1] snapshot/restore max abs diff: 0.00e+00
[2] all perturbation families step OK
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/__init__.py", line 78, in <module>
    from .tokenizer_processor import ActionTokenizerProcessorStep, TokenizerProcessorStep
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/tokenizer_processor.py", line 48, in <module>
    from transformers import AutoProcessor, AutoTokenizer
  File "/usr/local/lib/python3.13/dist-pa

## 1. Collect base-policy rollouts
Nominal (success-WM data + base success rate, expect roughly 0.6-0.7 for this checkpoint) and perturbed (failure-WM data).

In [46]:
!python scripts/01_collect.py --root $ROOT --n 500 --batch 128
for p in ['friction', 'cog', 'damping', 'action_noise']:
    !python scripts/01_collect.py --root $ROOT --n 200 --batch 128 --perturbation {p} --severity 2

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/__init__.py", line 78, in <module>
    from .tokenizer_processor import ActionTokenizerProcessorStep, TokenizerProcessorStep
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/tokenizer_processor.py", line 48, in <module>
    from transformers import AutoProcessor, AutoTokenizer
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 30, in <module>
    from . import dependenc

## 2. Train stand-in WMs (success + failure)

In [47]:
!python scripts/02_train_wm.py --root $ROOT --epochs 30

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Traceback (most recent call last):
  File "/content/code/scripts/02_train_wm.py", line 18, in <module>
    allnom = load_pickle(root / "data" / "rollouts_nominal.pkl")
  File "/content/code/dnh/experiment.py", line 85, in load_pickle
    with open(path, "rb") as f:
         ~~~~^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dnh_runs/data/rollouts_nominal.pkl'


## 3. RQ1: week-1 GO / NO-GO
Executes every one of K candidates from each branch state. **GO if the best within-state AUROC > 0.6.**
Start small (`--n-states 64`) to measure time, then scale.

In [48]:
!python scripts/03_rq1_ranking.py --root $ROOT --n-states 64 --K 16 --batch 256
!python scripts/03_rq1_ranking.py --root $ROOT --n-states 64 --K 16 --batch 256 --perturbation friction --severity 2

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/__init__.py", line 78, in <module>
    from .tokenizer_processor import ActionTokenizerProcessorStep, TokenizerProcessorStep
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/tokenizer_processor.py", line 48, in <module>
    from transformers import AutoProcessor, AutoTokenizer
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 30, in <module>
    from . import dependenc

## 4. Do-no-harm calibration (Algorithm 1)

In [49]:
!python scripts/04_calibrate_ltt.py --root $ROOT --scorer u --n-cal 200 --n-test 200 --K 16 --batch 200
!python scripts/04_calibrate_ltt.py --root $ROOT --scorer contrast --n-cal 200 --n-test 200 --K 16 --batch 200

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/__init__.py", line 78, in <module>
    from .tokenizer_processor import ActionTokenizerProcessorStep, TokenizerProcessorStep
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/tokenizer_processor.py", line 48, in <module>
    from transformers import AutoProcessor, AutoTokenizer
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 30, in <module>
    from . import dependenc

## 5. Success under shift at the certified setting

In [50]:
!python scripts/05_eval_shift.py --root $ROOT --scorer contrast --n 100 --K 16 --batch 100

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/__init__.py", line 78, in <module>
    from .tokenizer_processor import ActionTokenizerProcessorStep, TokenizerProcessorStep
  File "/usr/local/lib/python3.13/dist-packages/lerobot/processor/tokenizer_processor.py", line 48, in <module>
    from transformers import AutoProcessor, AutoTokenizer
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 30, in <module>
    from . import dependenc

In [51]:
import json, glob, pandas as pd
for f in sorted(glob.glob(f'{ROOT}/results/*.json')):
    print(f); print(json.dumps(json.load(open(f)), indent=1, default=str)[:1500]); print()